# Week 8 — End-to-End Business Analysis: Presentation Preparation
## Phase 2b SQL | PORA Academy Cohort 7 — **Exercises**

The demo walked through all five presentation deliverables end to end. This notebook is where your
group builds them — because the marks go to SQL you wrote, and a query you have only watched is not a
query you can defend when someone asks where the number came from.

It has two parts, and they work differently:

**Part A — Build three of the five deliverables yourself (Exercises 1–3).** Top 5 product categories by
revenue, the seller tier breakdown, and delivery performance against review scores. Each returns
**many rows**, so there is no single value a computer can tick. Instead each question lists the
verified **anchors** your output must line up with — if your top row or your headline figure matches
the anchor, the logic underneath it is sound.

**Part B — Verify your baseline (Questions 1–2).** Two queries with locked, curriculum-verified
answers and a check cell that prints ✅. Run them before every rehearsal and again before you present:
if these two come out right, your database loaded correctly and every figure you built on top of it
starts from solid ground. If they come out wrong, stop — nothing downstream is trustworthy, however
plausible it looks.

**How the cells work.** A Part A question is two cells: the question, then a blank `%%sql` cell whose
result displays straight below it. A Part B question is three cells: the question, a blank
`%%sql qN <<` answer cell, and a check cell. **Do not edit the check cells**, and use exactly the
column aliases each question asks for — the checks look columns up by name.

Run the setup cell first, then work top to bottom.

Run this cell first, as always. It rebuilds the SQLite database from the Olist CSVs and connects
the `%%sql` magic to it. Nothing below will run until it has finished.

In [ ]:
# =====================================================================
# Olist SQL Setup — runs on BOTH Google Colab and a local machine.
# Run this cell FIRST. It loads the 8 Olist tables into a SQLite
# database and connects the %%sql magic to it. You should not need to
# edit anything unless auto-detection fails (see the two knobs below).
#
# Design notes:
# - We teach SQL with the %%sql cell magic (jupysql), not pd.read_sql().
# - jupysql opens its OWN connection, so the DB must be a real FILE
#   (a :memory: DB would be invisible to it).
# - We use jupysql (the maintained SQL magic). On Colab we install it,
#   because Colab ships the legacy ipython-sql, which (a) can't take a
#   connection by engine variable and (b) renders every result through
#   prettytable.__dict__[style], crashing on modern prettytable with
#   KeyError 'DEFAULT'/'SINGLE_BORDER'. jupysql fixes both.
# - autopandas=True makes every %%sql result a pandas DataFrame, which
#   lets the self-check cells assert on .iloc/.shape directly.
# =====================================================================
import os, glob, sqlite3, tempfile, zipfile
import pandas as pd

# --- Optional knobs (leave blank; only set if auto-detect fails) ------
LOCAL_DATA_DIR = ""   # local run: folder that holds olist_orders_dataset.csv
DRIVE_ZIP_PATH = ""   # Colab: full path to phase-2-python-sql.zip in your Drive
# ---------------------------------------------------------------------

# Detect Colab (google.colab only imports there). Outside Colab — including
# the content-pipeline validator — this falls through to the local branch.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ON_COLAB = True
except ModuleNotFoundError:
    ON_COLAB = False


def _colab_find_zip():
    """Locate phase-2-python-sql.zip in Drive WITHOUT a full recursive scan
    (globbing '/content/drive/MyDrive/**' walks the entire Drive over the
    network and can hang for many minutes). Try explicit paths first, then a
    depth- and count-bounded breadth-first search that prints progress."""
    if DRIVE_ZIP_PATH:
        if os.path.exists(DRIVE_ZIP_PATH):
            return DRIVE_ZIP_PATH
        raise FileNotFoundError(f"DRIVE_ZIP_PATH is set but not found: {DRIVE_ZIP_PATH}")

    target = "phase-2-python-sql.zip"
    # Fast, instant checks of the most likely spots (top of Drive + course folder).
    for cand in (
        f"/content/drive/MyDrive/{target}",
        f"/content/drive/MyDrive/Data Analysis and AI Automation Course Cohort 7/Dataset/{target}",
        f"/content/{target}",
    ):
        if os.path.exists(cand):
            return cand

    # Bounded BFS: depth <= 4, at most ~600 folders, skipping hidden dirs.
    print("Searching your Google Drive for phase-2-python-sql.zip ...")
    root, queue, scanned = "/content/drive/MyDrive", [("/content/drive/MyDrive", 0)], 0
    while queue:
        d, depth = queue.pop(0)
        hit = os.path.join(d, target)
        if os.path.exists(hit):
            return hit
        if depth >= 4:
            continue
        try:
            for e in os.scandir(d):
                if e.is_dir() and not e.name.startswith("."):
                    queue.append((e.path, depth + 1))
        except OSError:
            continue
        scanned += 1
        if scanned % 50 == 0:
            print(f"  ...scanned {scanned} folders")
        if scanned >= 600:
            break

    raise FileNotFoundError(
        "Could not quickly find phase-2-python-sql.zip in your Drive. Put the zip at the "
        "TOP of your Drive (My Drive) and re-run, or set DRIVE_ZIP_PATH at the top of this "
        "cell to its exact path.")


def _find_csv_dir():
    """Return the folder that actually contains olist_orders_dataset.csv."""
    roots = []
    env_dir = os.environ.get("OLIST_DATA_PATH", "")   # set by the pipeline validator
    if env_dir:
        roots.append(env_dir)
    if LOCAL_DATA_DIR:
        roots.append(LOCAL_DATA_DIR)

    if ON_COLAB:
        extract_path = "/content/olist_data"
        # unzip only the first time; reuse the extracted CSVs afterwards
        if not glob.glob(f"{extract_path}/**/olist_orders_dataset.csv", recursive=True):
            zip_path = _colab_find_zip()
            os.makedirs(extract_path, exist_ok=True)
            print(f"Unzipping {os.path.basename(zip_path)} ...")
            with zipfile.ZipFile(zip_path) as z:
                z.extractall(extract_path)
        roots.append(extract_path)
    else:
        # Local: search cwd (recursively) + a few common spots — never the whole
        # home dir (that recursive walk can be very slow). Set LOCAL_DATA_DIR if
        # your CSVs live elsewhere.
        roots += [os.getcwd(),
                  os.path.expanduser("~/Downloads"),
                  os.path.expanduser("~/Desktop"),
                  os.path.expanduser("~/olist")]

    for root in roots:
        if os.path.exists(os.path.join(root, "olist_orders_dataset.csv")):
            return root
        hits = glob.glob(os.path.join(root, "**", "olist_orders_dataset.csv"), recursive=True)
        if hits:
            return os.path.dirname(hits[0])

    raise FileNotFoundError(
        "Olist CSVs not found. Set LOCAL_DATA_DIR (local) or DRIVE_ZIP_PATH (Colab) at "
        "the top of this cell.")


DATA_DIR = _find_csv_dir()
print("Data folder:", DATA_DIR)

# Build a file-based SQLite DB shared by pandas (loading) and jupysql (querying).
DB_PATH = os.environ.get("OLIST_DB_PATH") or (
    "/content/olist.db" if ON_COLAB else os.path.join(tempfile.gettempdir(), "olist.db"))

tables = {
    "orders": "olist_orders_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "product_category_translation": "product_category_name_translation.csv",
}

conn = sqlite3.connect(DB_PATH)
for table_name, filename in tables.items():
    df = pd.read_csv(os.path.join(DATA_DIR, filename))
    df.to_sql(table_name, conn, if_exists="replace", index=False)
    print(f"Loaded {table_name}: {len(df):,} rows")
conn.close()
print("\nDatabase ready.")

# On Colab, install jupysql so `%load_ext sql` loads it instead of the legacy
# ipython-sql (see header). Off Colab (local / pipeline validator) jupysql is
# already installed, so we skip the install and stay offline-safe.
if ON_COLAB:
    get_ipython().run_line_magic("pip", "install --quiet --upgrade jupysql")

get_ipython().run_line_magic("load_ext", "sql")

# Guard: if the legacy ipython-sql was already loaded earlier THIS session (e.g.
# an older cell ran first), the freshly installed jupysql cannot hot-swap in — a
# runtime restart is the only fix. jupysql exposes sql.connection.ConnectionManager;
# ipython-sql does not. Stop with a clear instruction instead of a later cryptic
# prettytable KeyError.
import sql.connection as _sqlconn
if not hasattr(_sqlconn, "ConnectionManager"):
    raise RuntimeError(
        "Legacy ipython-sql is active, not jupysql. On Colab: Runtime -> Restart session, "
        "then run THIS setup cell first (before any other cell). Locally: "
        "pip install --upgrade jupysql and restart the kernel."
    )

# Connect the %%sql magic to the SAME database file. autopandas=True is REQUIRED
# (see header). We connect with run_line_magic (not a literal `%sql` line) so the
# computed DB_PATH is interpolated correctly. Do NOT set SqlMagic.style.
get_ipython().run_line_magic("config", "SqlMagic.autopandas = True")
get_ipython().run_line_magic("config", "SqlMagic.feedback = 0")
get_ipython().run_line_magic("sql", f"sqlite:///{DB_PATH}")

# Verify (expected row counts — do not alter without re-running against data):
#   orders 99,441 | customers 99,441 | order_items 112,650 | order_payments 103,886
#   order_reviews 99,224 | products 32,951 | sellers 3,095 | product_category_translation 71


---
# Part A — Build the deliverables

Four rules apply to every question in this part, because they are the rules your presentation is
marked under:

1. **SQL only.** No pandas beyond displaying the result.
2. **Use a `WITH` block** whenever the query needs more than one step. Query readability carries 10 of
   the 40 marks, and a CTE is the cheapest way to earn them.
3. **Watch the grain.** `order_items`, `order_payments` and `order_reviews` each hold *many rows per
   order*. Join two of them directly and every `SUM`, `AVG` and `COUNT(*)` inflates — silently, with no
   error. If a question touches two of them, collapse one to one row per `order_id` in a CTE first.
4. **Check against an anchor before you believe a number.** Every question below gives you one.

## Exercise 1 — Top 5 product categories by revenue

*Presentation deliverable 2.*

Leadership wants to know where the money actually comes from. Write one query that returns the **five
highest-revenue product categories**, in English, one row per category:

| Alias | What it holds |
|---|---|
| `category` | the English category name |
| `product_revenue` | total `price` for that category, rounded to 2 dp |
| `order_count` | how many distinct orders contained a product in that category |
| `pct_of_categorised_revenue` | that category's share of all categorised revenue, rounded to 1 dp |

Sort by `product_revenue` descending and `LIMIT 5`.

**How to build it.** Revenue lives in `order_items.price`; the category name lives in `products`; the
English name lives in `product_category_translation`. That is two joins (Week 3) to get from a line
item to a readable label. Aggregate inside a `WITH` block so the share column can divide each
category's revenue by the total across the same block — and force REAL division with `* 100.0`, or
integer division hands you `0` for every row. One order can contain several line items, so count
orders with `COUNT(DISTINCT order_id)`.

**A judgement point worth a sentence on your slide.** Roughly **610 products** carry no category name
at all, and those rows fall out of an inner join. So your `product_revenue` column sums to *less* than
the platform's **R$13,591,643.70** of item revenue. That is why the share column is named
`pct_of_categorised_revenue` and not "% of revenue" — say which denominator you used before anyone
asks.

**Anchors — check your output against these:**
- Exactly 5 rows.
- The top row is **health_beauty** with **R$1,258,681.34** in product revenue. If your top row reads a
  different category, or a much larger number, you have probably joined a second many-per-order table
  and fanned the rows out.

In [ ]:
%%sql
-- Exercise 1 — top 5 product categories by revenue, English names.
-- Your query here
--
-- Sketch before you write:
--   WITH category_revenue AS ( ... one row per English category ... )
--   SELECT category, ROUND(...) AS product_revenue, order_count,
--          ROUND(... * 100.0 / (SELECT SUM(...) FROM category_revenue), 1)
--            AS pct_of_categorised_revenue
--   FROM category_revenue ORDER BY product_revenue DESC LIMIT 5

## Exercise 2 — Seller tier breakdown

*Presentation deliverable 3.*

Olist is a marketplace, so its revenue is really the sum of thousands of sellers — and those sellers
are nothing like each other. Use the Week 7 pattern to split them into **four equal-sized tiers by
revenue** and describe each tier. One row per tier, four rows in total:

| Alias | What it holds |
|---|---|
| `tier` | 1 to 4, where tier 1 is the highest-revenue quarter of sellers |
| `sellers` | how many sellers sit in that tier |
| `tier_revenue` | total revenue of the tier, rounded to 2 dp |
| `avg_seller_revenue` | average revenue per seller in that tier, rounded to 2 dp |
| `top_seller_in_tier` | the largest single seller revenue in that tier, rounded to 2 dp |
| `pct_of_seller_revenue` | the tier's share of all seller revenue, rounded to 1 dp |

Sort by `tier`.

**How to build it.** Two CTEs, stacked. The first collapses `order_items` to **one row per seller** —
`SUM(price)` as revenue, `COUNT(DISTINCT order_id)` as orders. The second adds
`NTILE(4) OVER (ORDER BY revenue DESC) AS tier` to that seller-level result, which is the whole trick:
`NTILE` cuts an ordered set into *n* buckets of equal size, so it must run **after** you are at seller
grain, never over raw line items. Then `GROUP BY tier` over the tiered CTE.

Why `ORDER BY revenue DESC` inside `NTILE` rather than ascending? So that tier 1 means *biggest*,
which is how you will say it out loud. Order it the other way and your slide's tier numbers read
backwards from your commentary.

**Anchors — check your output against these:**
- Exactly 4 rows, with the four `sellers` counts near-identical — that is what "equal-sized tiers"
  means, and it is the fastest way to spot that `NTILE` ran at the wrong grain.
- `top_seller_in_tier` for tier 1 is **R$229,472.63** — the single best-performing seller on the
  platform.
- `pct_of_seller_revenue` sums to 100.0 across the four rows.

**Then answer in one sentence:** how much of the platform's seller revenue does tier 1 carry — and what
would it mean for Olist if a handful of those sellers left?

In [ ]:
%%sql
-- Exercise 2 — seller tiers via NTILE, one row per tier.
-- Your query here
--
-- Sketch before you write:
--   WITH seller_revenue AS ( ... one row per seller_id ... ),
--        tiered_sellers  AS ( SELECT ..., NTILE(4) OVER (ORDER BY revenue DESC) AS tier
--                             FROM seller_revenue )
--   SELECT tier, ... FROM tiered_sellers GROUP BY tier ORDER BY tier

## Exercise 3 — Delivery performance, and what it does to review scores

*Presentation deliverable 4.*

This is the deliverable that connects an operational number to a customer-facing one, which is exactly
what leadership listens for. Write **one query** returning a single row:

| Alias | What it holds |
|---|---|
| `delivered_orders` | how many orders were actually delivered, with a real delivery date |
| `avg_delivery_days` | average days from purchase to delivery, rounded to 1 dp |
| `late_orders` | how many arrived after the estimated delivery date |
| `late_pct` | `late_orders` as a percentage of `delivered_orders`, rounded to 1 dp |
| `avg_score_late` | average review score of the late orders, rounded to 2 dp |
| `avg_score_on_time` | average review score of the on-time orders, rounded to 2 dp |

Filter to `order_status = 'delivered'` **and** `order_delivered_customer_date IS NOT NULL` — an order
that never arrived has no delivery duration to average, and `IS NOT NULL` is the only way to test that
(`!= NULL` silently matches nothing).

**Three mechanics to get right.** SQLite has no date type and no `DATEDIFF`, so a duration in days is
`julianday(delivered) - julianday(purchased)` over TEXT timestamps. A count-alongside-other-aggregates
is `SUM(CASE WHEN ... THEN 1 ELSE 0 END)`; the *late* condition is that the delivered date is greater
than the estimated date. And `late_pct` needs `* 100.0`, not `* 100`.

**The grain trap, and the one that matters most this week.** `order_reviews` holds **many rows per
order**, so you cannot join it to a delivered-orders set and then average freely. Collapse reviews to
one row per `order_id` in a CTE first — `AVG(review_score) GROUP BY order_id` — then `LEFT JOIN` that
one-row-per-order result onto your delivery CTE. `LEFT`, not inner: an order with no review still had a
delivery time, and dropping it would quietly bias your average.

The two split scores come from `AVG(CASE WHEN is_late = 1 THEN review_score END)` — with no `ELSE`, so
the other group's rows become NULL and `AVG` ignores them rather than counting them as zero.

**Anchors — check your output against these:**
- `avg_delivery_days` is **12.6**.
- `late_orders` is **7,826**, which is **8.1%** of delivered orders.
- `avg_score_late` should land clearly **below** `avg_score_on_time`. The size of that gap is your
  slide.

**Then say the two cautions out loud when you present:** this is *association*, not proof of cause, and
every row here is a delivered order — the customers whose orders never arrived are excluded entirely,
and they are unlikely to be the happy ones.

In [ ]:
%%sql
-- Exercise 3 — delivery headline plus the late/on-time review gap.
-- Your query here
--
-- Sketch before you write:
--   WITH rev AS ( ... AVG(review_score) per order_id ... ),
--        del AS ( ... one row per delivered order: delivery_days + is_late flag ... )
--   SELECT ... FROM del d LEFT JOIN rev r ON d.order_id = r.order_id

---
# Part B — Verify your baseline before you present

The two questions below have locked, curriculum-verified answers, and they are the most practical
cells in this notebook. They touch no joins and no CTEs, so nothing about them can fan out: they are a
straight read of the two figures every other number on your slides is built from.

Run them **before every rehearsal and immediately before you present**. Two green ticks mean the data
loaded correctly. They are also step 3 of the DeepSeek protocol — when the AI drafts a query for you,
make it reproduce a number you already trust before you believe a number you don't.

Each question is three cells: the question, a blank `%%sql qN <<` answer cell, and a check cell that
prints ✅. The column aliases matter — the check cells look them up by name.

## Question 1 — Platform overview KPIs

*Presentation deliverable 1 — and the first thing on your first slide.*

Write a **single query** over the `orders` table returning one row with four columns:

| Alias | What it holds |
|---|---|
| `total_orders` | the count of distinct `order_id` values |
| `delivered` | how many orders have `order_status = 'delivered'` |
| `canceled` | how many orders have `order_status = 'canceled'` |
| `delivered_pct` | delivered as a percentage of all orders, rounded to 1 dp |

Do it **without a `WHERE` clause** — that is the point of the exercise.
`SUM(CASE WHEN ... THEN 1 ELSE 0 END)` counts the rows matching a condition while still counting every
other row too, which is how one pass over the table yields three different counts. For
`delivered_pct`, force REAL division with `1.0` on the `THEN` branch, or integer division returns `0`.

**Expected:** 99,441 total orders · 96,478 delivered · 625 canceled · 97.0% delivered

In [ ]:
%%sql q1 <<
-- Your query here

In [ ]:
# --- CHECK Q1 — do not edit ---
assert int(q1.iloc[0]['total_orders']) == 99441, "Q1: expected 99,441 total orders"
assert int(q1.iloc[0]['delivered']) == 96478, "Q1: expected 96,478 delivered orders"
assert int(q1.iloc[0]['canceled']) == 625, "Q1: expected 625 canceled orders"
assert round(float(q1.iloc[0]['delivered_pct']), 1) == 97.0, "Q1: expected delivered_pct of 97.0"
print("✅ Q1 correct")
q1  # show the result of your query

## Question 2 — Total GMV verification

Gross merchandise value is the headline money figure in your presentation, so it is the one you least
want to get wrong. Write a **single query** over `order_items` returning one row with three columns:

| Alias | What it holds |
|---|---|
| `product_revenue` | the sum of `price`, rounded to 2 dp |
| `freight_revenue` | the sum of `freight_value`, rounded to 2 dp |
| `total_gmv` | the sum of `price + freight_value`, rounded to 2 dp |

**Use `order_items` alone.** It is tempting to join `orders` or `order_payments` "for completeness",
and that is precisely the trap: `order_items` and `order_payments` are both many-per-order, so joining
them repeats every item row once per payment row and inflates the total. No error is raised. The query
runs, the number looks plausible, and it is wrong. The money question only ever needed `order_items`,
at item grain.

**Expected:** product revenue R$13,591,643.70 · freight revenue R$2,251,909.54 · total GMV
R$15,843,553.24

In [ ]:
%%sql q2 <<
-- Your query here

In [ ]:
# --- CHECK Q2 — do not edit ---
assert round(float(q2.iloc[0]['product_revenue']), 2) == 13591643.70, \
    "Q2: expected product_revenue of 13,591,643.70"
assert round(float(q2.iloc[0]['freight_revenue']), 2) == 2251909.54, \
    "Q2: expected freight_revenue of 2,251,909.54"
assert round(float(q2.iloc[0]['total_gmv']), 2) == 15843553.24, \
    "Q2: expected total_gmv of 15,843,553.24"
print("✅ Q2 correct")
q2  # show the result of your query

---
## Deliverable 5 — your group's own business question

The fifth deliverable has no model answer, no anchor and no check cell, because the question is yours
to choose. That is deliberate: the first four deliverables test whether you can write SQL, and this one
tests whether you can decide what is worth asking.

Pick a question leadership would actually act on, and that the Olist tables can genuinely answer. Then
build it in the cell below, in three deliberate steps:

1. **Write the question as a sentence first**, in a comment, before any SQL. If you cannot state it in
   one line, the query will not have a clear shape either.
2. **Decide the grain.** Order? Seller? Category? Customer state? Month? Name it, then collapse to it
   in a `WITH` block. Most wrong capstone numbers are grain mistakes, not syntax mistakes.
3. **Give the reader something to judge against** — a share-of-total column, a benchmark from a window
   function, or a second period to compare with. A bare number invites "is that good?"; a number beside
   a benchmark answers it.

**Before it goes on a slide, sanity-check it.** You have no verified anchor for this one, so make your
own: does the total reconcile with a figure you *do* trust? Does the row count match what the grain
implies? Would the number still look right if you computed it a second, different way? If the query
touches two of `order_items`, `order_payments` or `order_reviews`, assume it is inflated until you have
proved otherwise.

And write the one sentence of meaning that goes underneath it. Leadership will not read your SQL — they
will read the number and your sentence.

In [ ]:
%%sql
-- Deliverable 5 — your group's chosen question. Replace this placeholder.
-- Step 1 — state the question in one line:
--   "..."
-- Step 2 — name the grain, then collapse to it:  WITH ... AS ( ... )
-- Step 3 — present it, with a share or benchmark column so the reader can judge it.
SELECT 'write your group question here' AS todo

---
## Before you present

Work back through what you now have. Exercises 1–3 are deliverables 2, 3 and 4; Questions 1–2 are
deliverable 1 and the money anchor sitting behind all of it; the cell above is deliverable 5. That is
the whole presentation, and every figure in it is one you produced yourself.

A last rehearsal checklist:

- **Run Part B.** Two green ticks, every time, before the deck is opened.
- **Re-check the three Part A anchors**: health_beauty at **R$1,258,681.34**, the top seller at
  **R$229,472.63**, delivery averaging **12.6 days** with **7,826** late arrivals (**8.1%**). Matching
  anchors are what let you answer "where did that come from?" in one breath.
- **One sentence per number.** Ten minutes across five deliverables is two minutes each — one slide,
  one query, one sentence of meaning. Write the sentences before you write the slides.
- **Know your denominators and your exclusions.** Which revenue total, which set of orders, which year.
  The question you cannot answer is the one that costs marks.
- **Be able to re-run any query live.** If a number cannot be reproduced on demand, leave it off the
  slide.